# Cross-experiment analyses and plots

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
import logging
logging.getLogger('matplotlib.font_manager').disabled = True

import helpers as h
from settings import data_dir, out_dir
og_data_dir = data_dir

## Configuration

In [ ]:
# ============================================================================
# USER CONFIG: edit `version_dirs` to choose which experiment variants to
# compare (each must be a key in version_names/version_models/version_suffixes
# below, and a subfolder under `out_dir`, e.g. `out_dir/V2/<family>/<model>/`).
# The dictionaries below are examples from our own experiments (see
# README.md); add/remove entries and models to match your own data.
# ============================================================================
version_dirs = ["", "V3/", "V4/"]

# Base or CoT?
model_set = "base"

# Version display names (for plot labels, dict keys, etc.)
version_names = {
    "": "Alchemy",
    "V2/": "Persona",
    "V3/": "Skill acquisition",
    "V4/": "Non-words",
    "V5/": "CW 48",
    "V6/": "CW 120",
    "V7/": "CW 24",
    "V8/": "CW 12",
}

# Models available per version
version_models = {
    "": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": ["gemini-2.5-pro_cot", "gpt-5_cot", "qwen3-32b_cot"],
    },
    "V2/": {
        "base": ["gemini-2.5-pro", "gpt-5", "qwen3-32b", "centaur"],
        "cot": [],
    },
    "V3/": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": [],
    },
    "V4/": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": [],
    },
    "V5/": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": [],
    },
    "V6/": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": [],
    },
    "V7/": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": [],
    },
    "V8/": {
        "base": ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"],
        "cot": [],
    },
}

# Version suffixes (for file naming)
version_suffixes = {
    "": "_alchemy",
    "V2/": "_persona",
    "V3/": "_acrobat",
    "V4/": "_nonwords",
    "V5/": "_cw48",
    "V6/": "_cw120",
    "V7/": "_cw24",
    "V8/": "_cw12",
}

# Temperature to use across all versions
temp = 1.0

plot_version_suffix = "_vs".join([version_suffixes[v] for v in version_dirs])


In [ ]:
model_colors_dict = {
    "gpt-5": "#12a37f",
    "gpt-5_cot": "#12a37f",
    "gemini-2.5-pro": "#F4B400",
    "gemini-2.5-pro_cot": "#F4B400",
    "centaur": "#691a5e",
    "claude-sonnet-4.5": "#da7756",
    "qwen3-32b": "#5a52d9",
    "qwen3-32b_cot": "#5a52d9",
}

model_markers_dict = {
    "human": "X",
    "gpt-5": "*",
    "gpt-5_cot": "*",
    "gemini-2.5-pro": "D",
    "gemini-2.5-pro_cot": "D",
    "centaur": "h",
    "claude-sonnet-4.5": "^",
    "qwen3-32b": "P",
    "qwen3-32b_cot": "P",
}

model_names_dict = {
    "human": "Humans",
    "gpt-5": "GPT-5",
    "gpt-5_cot": "GPT-5 CoT",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "gemini-2.5-pro_cot": "Gemini 2.5 Pro CoT",
    "centaur": "Centaur",
    "claude-sonnet-4.5": "Claude Sonnet 4.5",
    "qwen3-32b": "Qwen 32B",
    "qwen3-32b_cot": "Qwen 32B CoT",
}

In [ ]:
from helpers import adjust_color_brightness
from matplotlib.patches import Patch

# Version index 0 (base/"") keeps the original color from model_colors_dict.
# Subsequent versions get progressively lighter.
def get_version_shade(base_color, version_idx, n_versions, lighten_step=0.35):
    """Return shade of base_color: version_idx=0 is original, higher indices are lighter."""
    if version_idx == 0 or n_versions <= 1:
        return base_color
    factor = 1 + version_idx * lighten_step
    return adjust_color_brightness(base_color, factor)


def make_cross_exp_legend(ax, all_models, version_dirs, version_names,
                          model_colors_dict, model_markers_dict, model_names_dict,
                          n_versions, fontsize=12, ncol=3, loc="upper right",
                          bbox_to_anchor=None, show_versions=True):
    """Build a legend: model identities (line+marker) + optionally version key (grey boxes)."""
    handles, labels = [], []

    # Part 1: human (line + marker)
    handles.append(Line2D([0], [0], marker="X", color="k",
                          markerfacecolor="white", markeredgecolor="k",
                          markersize=7, linewidth=2))
    labels.append("Humans")

    # Part 1: one entry per model (line + marker in model color)
    for model in all_models:
        h = Line2D([0], [0], marker=model_markers_dict[model],
                   color=model_colors_dict[model],
                   markerfacecolor="white",
                   markeredgecolor=model_colors_dict[model],
                   markersize=7, linewidth=2)
        handles.append(h)
        labels.append(model_names_dict[model])

    # Part 2: version key (grey boxes, no border)
    if show_versions:
        for v_idx, vd in enumerate(version_dirs):
            vname = version_names[vd]
            grey_shade = get_version_shade("#555555", v_idx, n_versions, lighten_step=0.5)
            display_name = "Alchemy" if vd == "" else vname.capitalize()
            handles.append(Patch(facecolor=grey_shade, edgecolor="none"))
            labels.append(display_name)

    legend_kwargs = dict(fontsize=fontsize, ncol=ncol,
                         handletextpad=0.5, columnspacing=1.0)
    if bbox_to_anchor is not None:
        legend_kwargs["bbox_to_anchor"] = bbox_to_anchor
    legend_kwargs["loc"] = loc

    ax.legend(handles, labels, **legend_kwargs)

## Gather data

In [ ]:
plots_dir = "results/figures/"
filled_markers = Line2D.filled_markers[::-1]

# Human data (shared across all versions)
human_out_dir = f"{out_dir}/human/"

human_task_df = pd.read_csv(f"{human_out_dir}task_df.csv")
human_train_df = pd.read_csv(f"{human_out_dir}train_df.csv")
human_learn_df = pd.read_csv(f"{human_out_dir}learn_df.csv")
human_test_df = pd.read_csv(f"{human_out_dir}test_df.csv")

human_plotter = h.Plotter(
    task_df=human_task_df,
    train_df=human_train_df,
    learn_df=human_learn_df,
    test_df=human_test_df,
)
human_plotter.main_color = "k"
human_plotter.marker = filled_markers[0]
human_plotter.get_metrics()

In [ ]:
# plotters dict: keys are (model, version_name) tuples
# Human is stored as ("human", "human") since it's shared across versions
plotters = {}
plotters[("human", "human")] = human_plotter

for version_dir in version_dirs:
    version_name = version_names[version_dir]

    # Resolve data_dir for this version
    if version_dir == "V3/":
        v_data_dir = og_data_dir.replace("Alchemist", "Acrobat")
        version_dir_for_data_dir = ""
    elif version_dir == "V4/":
        v_data_dir = og_data_dir.replace("Alchemist", "NonWords")
        version_dir_for_data_dir = ""
    else:
        v_data_dir = og_data_dir
        version_dir_for_data_dir = version_dir

    base_models = version_models[version_dir]["base"]
    cot_models = version_models[version_dir]["cot"]

    if temp == 0.0:
        base_models = [m for m in base_models if m in ["gemini-2.5-pro", "claude-sonnet-4.5", "qwen3-32b"]]
        cot_models = []

    for model in version_models[version_dir][model_set]:
        family = h.get_family(model)

        task_df = pd.read_csv(f"{out_dir}{version_dir}/{family}/{model}/task_df.csv")
        train_df = pd.read_csv(f"{out_dir}{version_dir}/{family}/{model}/train_df.csv")
        learn_df = pd.read_csv(f"{out_dir}{version_dir}/{family}/{model}/learn_df.csv")
        test_df = pd.read_csv(f"{out_dir}{version_dir}/{family}/{model}/test_df.csv")
        temp_vals = task_df.temperature.unique()
        if len(temp_vals) <= 1:
            print(f"[{version_name}] {model} only has one temperature setting: {temp_vals[0]}.")
            temp_plotter = h.Plotter(task_df, train_df, learn_df, test_df)
        else:
            print(f"[{version_name}] {model} has multiple temperature settings: {temp_vals}.")
            print(f"    Only processing temperature={temp}.")
            temp_plotter = h.Plotter(
                task_df.query("temperature==@temp"),
                train_df.query("phase==0 & temperature==@temp"),
                learn_df.query("phase==1 & temperature==@temp"),
                test_df.query("phase==2 & temperature==@temp"),
            )

        temp_plotter.main_color = model_colors_dict[model]
        temp_plotter.marker = model_markers_dict[model]
        temp_plotter.get_metrics()
        plotters[(model, version_name)] = temp_plotter

## Plots

### Performance averages

In [ ]:
# Performance metrics to show as bars
perf_metrics = {
    "p_correct_b1": "Practice",
    "early_learning": "Early learning\n(blocks 1-2)",
    "late_learning": "Late learning\n(blocks 5-6)",
    "p_correct_b8_base": "Test\n(in distribution)",
    "p_correct_b8_infer": "Test\n(out of distribution)",
}

# Performance CDF metrics (columns 1+ after the learning curve)
perf_cdf_metrics = {
    "early_learning": "Early learning (blocks 1-2)",
    "late_learning": "Late learning (blocks 5-6)",
    "p_correct_b8_base": "Test (in distribution)",
    "p_correct_b8_infer": "Test (out of distribution)",
}

# Collect all models across selected versions (preserving order)
all_models = []
for vd in version_dirs:
    for m in version_models[vd][model_set]:
        if m not in all_models:
            all_models.append(m)

n_versions = len(version_dirs)
n_cols = 1 + len(perf_cdf_metrics)  # learning curve / practice + 4 CDF metrics
n_version_rows = len(version_dirs)
n_cdf_rows = 1 + n_version_rows  # human row + one per version
n_total_rows = 1 + n_cdf_rows  # bar row + CDF rows
block_labels = ["Practice"] + [str(b) for b in range(1, 7)] + ["Test"]

# Bar layout
group_width = 0.8
bar_width = group_width / n_versions
version_offsets = [(v - (n_versions - 1) / 2) * bar_width for v in range(n_versions)]

fig, all_axs = plt.subplots(n_total_rows, n_cols,
                            figsize=(n_cols * 5, n_total_rows * 3.5),
                            gridspec_kw={"height_ratios": [1] + [1] * n_cdf_rows})

# ==================== ROW 0: BAR CHARTS ====================
hp = plotters[("human", "human")]
for mi, (metric_key, metric_label) in enumerate(perf_metrics.items()):
    ax = all_axs[0, mi]
    ax.set_title(metric_label, fontsize=16)

    # Human bar
    if metric_key == "early_learning":
        vals = np.mean([hp.metrics["p_correct_b2"], hp.metrics["p_correct_b3"]], axis=0)
    elif metric_key == "late_learning":
        vals = np.mean([hp.metrics["p_correct_b6"], hp.metrics["p_correct_b7"]], axis=0)
    else:
        vals = hp.metrics[metric_key]
    ax.bar(0, np.mean(vals), yerr=np.std(vals) / np.sqrt(hp.n_subjs),
           color="k", width=group_width)

    # Model bars
    for m_idx, model in enumerate(all_models):
        group_center = m_idx + 1
        for v_idx, vd in enumerate(version_dirs):
            vname = version_names[vd]
            key = (model, vname)
            if key not in plotters:
                continue
            plotter = plotters[key]
            if metric_key == "early_learning":
                vals = np.mean([plotter.metrics["p_correct_b2"], plotter.metrics["p_correct_b3"]], axis=0)
            elif metric_key == "late_learning":
                vals = np.mean([plotter.metrics["p_correct_b6"], plotter.metrics["p_correct_b7"]], axis=0)
            else:
                vals = plotter.metrics[metric_key]
            shade = get_version_shade(plotter.main_color, v_idx, n_versions)
            x = group_center + version_offsets[v_idx]
            ax.bar(x, np.mean(vals), yerr=np.std(vals) / np.sqrt(plotter.n_subjs),
                   color=shade, width=bar_width * 0.9)

    ax.set_ylim(0.0, 1.05)
    ax.set_xlim(-0.5, len(all_models) + 0.5)
    ax.set_xticks([])
    if mi == 0:
        ax.set_ylabel("P(correct)", fontsize=16)

# ==================== ROW 1: HUMAN CDFs ====================
# Learning curve (col 0)
lc_means = np.array([np.mean(hp.metrics[f"p_correct_b{b}"]) for b in range(1, 9)])
lc_sems = np.array([np.std(hp.metrics[f"p_correct_b{b}"]) / np.sqrt(hp.n_subjs) for b in range(1, 9)])
all_axs[1, 0].fill_between(np.arange(1, 9), lc_means - lc_sems, lc_means + lc_sems, color="k", alpha=0.3)
all_axs[1, 0].plot(np.arange(1, 9), lc_means, color="k", marker="X",
                   mec="k", mfc="white", markersize=6, label="Humans")
all_axs[1, 0].set_title("Learning curve", fontsize=16)
all_axs[1, 0].set_ylim(0.0, 1.05)
all_axs[1, 0].set_ylabel("Humans\nP(correct)", fontsize=14)
all_axs[1, 0].set_xticks(np.arange(1, 9))
all_axs[1, 0].set_xticklabels([])

# CDFs (cols 1+)
for col, (metric_key, metric_label) in enumerate(perf_cdf_metrics.items()):
    ax = all_axs[1, col + 1]
    if metric_key == "early_learning":
        vals = np.mean([hp.metrics["p_correct_b2"], hp.metrics["p_correct_b3"]], axis=0)
    elif metric_key == "late_learning":
        vals = np.mean([hp.metrics["p_correct_b6"], hp.metrics["p_correct_b7"]], axis=0)
    else:
        vals = hp.metrics[metric_key]
    y = np.sort(vals)
    x = np.arange(1, len(y) + 1) / len(y)
    ax.plot(x, y, color="k", label="Humans",
            mec="k", mfc="white", markersize=6, marker="X")
    ax.set_title(metric_label, fontsize=16)
    ax.set_ylim(0.0, 1.05)
    if col == 0:
        ax.set_ylabel("P(correct)", fontsize=14)
    ax.set_xticklabels([])

# ==================== ROWS 2+: LLM CDFs ====================
for row_idx, vd in enumerate(version_dirs):
    vname = version_names[vd]
    row = row_idx + 2  # offset by bar row + human row
    models_in_version = version_models[vd][model_set]

    # Learning curve (col 0)
    ax_lc = all_axs[row, 0]
    for model in models_in_version:
        key = (model, vname)
        if key not in plotters:
            continue
        plotter = plotters[key]
        lc_means = np.array([np.mean(plotter.metrics[f"p_correct_b{b}"]) for b in range(1, 9)])
        lc_sems = np.array([np.std(plotter.metrics[f"p_correct_b{b}"]) / np.sqrt(plotter.n_subjs) for b in range(1, 9)])
        ax_lc.fill_between(np.arange(1, 9), lc_means - lc_sems, lc_means + lc_sems,
                           color=plotter.main_color, alpha=0.3)
        ax_lc.plot(np.arange(1, 9), lc_means, color=plotter.main_color,
                   marker=plotter.marker, mec=plotter.main_color, mfc="white",
                   markersize=6, label=model_names_dict[model])
    ax_lc.set_ylim(0.0, 1.05)
    ax_lc.set_ylabel(f"{vname}\nP(correct)", fontsize=14)
    ax_lc.set_xticks(np.arange(1, 9))
    if row == n_total_rows - 1:
        ax_lc.set_xticklabels(block_labels, fontsize=10)
        ax_lc.set_xlabel("Block number", fontsize=14)
    else:
        ax_lc.set_xticklabels([])

    # CDFs (cols 1+)
    for col, (metric_key, metric_label) in enumerate(perf_cdf_metrics.items()):
        ax = all_axs[row, col + 1]
        for model in models_in_version:
            key = (model, vname)
            if key not in plotters:
                continue
            plotter = plotters[key]
            if metric_key == "early_learning":
                vals = np.mean([plotter.metrics["p_correct_b2"], plotter.metrics["p_correct_b3"]], axis=0)
            elif metric_key == "late_learning":
                vals = np.mean([plotter.metrics["p_correct_b6"], plotter.metrics["p_correct_b7"]], axis=0)
            else:
                vals = plotter.metrics[metric_key]
            y = np.sort(vals)
            x = np.arange(1, len(y) + 1) / len(y)
            ax.plot(x, y, color=plotter.main_color, label=model_names_dict[model],
                    mec=plotter.main_color, mfc="white", markersize=6, marker=plotter.marker)
        ax.set_ylim(0.0, 1.05)
        if col == 0:
            ax.set_ylabel("P(correct)", fontsize=14)
        if row == n_total_rows - 1:
            ax.set_xlabel("Normalized subject number", fontsize=14)
        else:
            ax.set_xticklabels([])

# Legend inside top-left subplot
make_cross_exp_legend(all_axs[0, 0], all_models, version_dirs, version_names,
                      model_colors_dict, model_markers_dict, model_names_dict,
                      n_versions, fontsize=11, ncol=2, loc="upper left")

fig.tight_layout()

# Border around the learning curve column (rows 1+)
fig.canvas.draw()
top_bbox = all_axs[1, 0].get_position()
bot_bbox = all_axs[n_total_rows - 1, 0].get_position()
pad_x = 0.02
pad_y = 0.03
rect = plt.Rectangle(
    (top_bbox.x0 - pad_x * 2 + 0.003, bot_bbox.y0 - pad_y - 0.005),
    top_bbox.width + 2 * pad_x,
    top_bbox.y1 - bot_bbox.y0 + 2 * pad_y,
    transform=fig.transFigure,
    linewidth=2,
    edgecolor="k",
    facecolor="none",
    clip_on=False,
    zorder=1000,
)
fig.add_artist(rect)

fig.savefig(f"{plots_dir}cross_exp_{model_set}_models_humans_vs{plot_version_suffix}_performance.png", dpi=300, bbox_inches="tight")

### Goal selection distributions (CDF)

In [ ]:
goal_metrics = {
    "p_2act_goals": "P(2-action goal chosen)",
    "p_repeat_goal_1back": "P(repeat goal)",
    "goal_entropy": "Goal entropy",
    "fav_goal_position": "Most chosen goal position",
    "n_max_any_0_6_pattern": "N. of goal rotation patterns",
    "n_systematic_hypothesis_testing": "N. of systematic actions\nby position",
}

goal_metric_titles = {
    "p_2act_goals": "How often are 2-action goals chosen?",
    "p_repeat_goal_1back": "How often is the same goal repeated?",
    "goal_entropy": "How often and widely do goals vary?",
    "fav_goal_position": "Which goal position\nis chosen most often?",
    "n_max_any_0_6_pattern": "How often is there\na goal rotation pattern?",
    "n_systematic_hypothesis_testing": "Trying out permutations in order",
}

n_metric_cols = len(goal_metrics)
n_version_rows = len(version_dirs)
n_cdf_rows = 1 + n_version_rows  # human row + one row per version
n_total_rows = 1 + n_cdf_rows  # bar row + CDF rows

# Bar layout
group_width = 0.8
bar_width = group_width / n_versions
version_offsets = [(v - (n_versions - 1) / 2) * bar_width for v in range(n_versions)]

fig, axs = plt.subplots(n_total_rows, n_metric_cols,
                        figsize=(n_metric_cols * 5, n_total_rows * 3.5),
                        gridspec_kw={"height_ratios": [1] + [1] * n_cdf_rows})

hp = plotters[("human", "human")]

# ==================== ROW 0: BAR CHARTS ====================
for mi, (metric_key, metric_label) in enumerate(goal_metrics.items()):
    ax = axs[0, mi]
    ax.set_title(goal_metric_titles[metric_key], fontsize=16)

    bar_min, bar_max = np.inf, -np.inf

    # Human bar
    h_vals = hp.metrics[metric_key]
    h_mean = np.mean(h_vals)
    h_sem = np.std(h_vals) / np.sqrt(hp.n_subjs)
    ax.bar(0, h_mean, yerr=h_sem, color="k", width=group_width)
    bar_min = min(bar_min, h_mean - h_sem)
    bar_max = max(bar_max, h_mean + h_sem)

    # Model bars (grouped by model, shaded by version)
    for m_idx, model in enumerate(all_models):
        group_center = m_idx + 1
        for v_idx, vd in enumerate(version_dirs):
            vname = version_names[vd]
            key = (model, vname)
            if key not in plotters:
                continue
            plotter = plotters[key]
            vals = plotter.metrics[metric_key]
            m_mean = np.mean(vals)
            m_sem = np.std(vals) / np.sqrt(plotter.n_subjs)
            shade = get_version_shade(plotter.main_color, v_idx, n_versions)
            x = group_center + version_offsets[v_idx]
            ax.bar(x, m_mean, yerr=m_sem, color=shade, width=bar_width * 0.9)
            bar_min = min(bar_min, m_mean - m_sem)
            bar_max = max(bar_max, m_mean + m_sem)

    margin = 0.05 * (bar_max - bar_min) if bar_max > bar_min else 0.1
    y_lo = min(0, bar_min - margin)
    ax.set_ylim(y_lo, bar_max + margin)
    ax.set_xlim(-0.5, len(all_models) + 0.5)
    ax.set_xticks([])
    ax.set_ylabel(metric_label, fontsize=14)

# ==================== ROW 1: HUMAN CDFs ====================
for col, (metric_key, metric_label) in enumerate(goal_metrics.items()):
    ax = axs[1, col]
    vals = hp.metrics[metric_key]
    y = np.sort(vals)
    x = np.arange(1, len(y) + 1) / len(y)
    ax.plot(x, y, color="k", label="Humans",
            mec="k", mfc="white", markersize=6, marker="X")
    min_val, max_val = y.min(), y.max()
    ax.set_ylim(min_val - 0.05 * (max_val - min_val), max_val + 0.05 * (max_val - min_val))
    if col == 0:
        ax.set_ylabel("Humans\n" + metric_label, fontsize=14)
    else:
        ax.set_ylabel(metric_label, fontsize=14)
    ax.set_xticklabels([])

# ==================== ROWS 2+: LLM CDFs ====================
for row_idx, vd in enumerate(version_dirs):
    vname = version_names[vd]
    row = row_idx + 2  # offset by bar row + human row
    models_in_version = version_models[vd][model_set]

    for col, (metric_key, metric_label) in enumerate(goal_metrics.items()):
        ax = axs[row, col]
        global_min, global_max = np.inf, -np.inf
        for model in models_in_version:
            key = (model, vname)
            if key not in plotters:
                continue
            plotter = plotters[key]
            vals = plotter.metrics[metric_key]
            global_min = min(global_min, vals.min())
            global_max = max(global_max, vals.max())
            y = np.sort(vals)
            x = np.arange(1, len(y) + 1) / len(y)
            ax.plot(x, y, color=plotter.main_color, label=model_names_dict[model],
                    mec=plotter.main_color, mfc="white", markersize=6, marker=plotter.marker)
        # Also include human range for consistent y-limits
        h_vals = hp.metrics[metric_key]
        global_min = min(global_min, h_vals.min())
        global_max = max(global_max, h_vals.max())
        margin = 0.05 * (global_max - global_min) if global_max > global_min else 0.1
        ax.set_ylim(global_min - margin, global_max + margin)
        if col == 0:
            ax.set_ylabel(f"{vname}\n{metric_label}", fontsize=14)
        else:
            ax.set_ylabel(metric_label, fontsize=14)
        if row == n_total_rows - 1:
            ax.set_xlabel("Normalized subject number", fontsize=14)
        else:
            ax.set_xticklabels([])

# Legend inside top-right subplot
make_cross_exp_legend(axs[0, 4], all_models, version_dirs, version_names,
                      model_colors_dict, model_markers_dict, model_names_dict,
                      n_versions, fontsize=11, ncol=1, loc="upper right")

fig.tight_layout()
fig.savefig(f"{plots_dir}cross_exp_{model_set}_models_humans_vs{plot_version_suffix}_goal_selection_cdf.png", dpi=300, bbox_inches="tight")

In [ ]:
f"{plots_dir}cross_exp_{model_set}_models_humans_vs{plot_version_suffix}_goal_selection_cdf.png"

## Quantitative comparisons

Compare each model across two experiment versions (median via MWU, distribution via Energy/KS/$\chi^2$).

In [ ]:
# Select which two versions to compare
version_A = version_dirs[0]
version_B = version_dirs[1]

vname_A = version_names[version_A]
vname_B = version_names[version_B]

# Models present in both versions
compare_models = [m for m in version_models[version_A][model_set]
                  if m in version_models[version_B][model_set]]

print(f"Comparing: {vname_A} vs {vname_B}")
print(f"Models: {compare_models}")

In [ ]:
from scipy import stats
from hyppo.ksample import Energy
from IPython.display import display, HTML

prefixes = {
    "p_correct_learning": "P(correct) at learning",
    "p_correct_b8_base": "P(correct) at testing\n(in distribution)",
    "p_correct_b8_infer": "P(correct) at testing\n(out of distribution)",
    "p_2act_goals": "P(2-action goal chosen)",
    "p_repeat_goal_1back": "P(repeat goal)",
    "goal_entropy": "Goal entropy",
    "fav_goal_position": "Most chosen goal position",
    "n_max_any_0_6_pattern": "N. of goal rotation patterns",
    "n_systematic_hypothesis_testing": "N. of systematic actions\nby position",
}

cat_vars = ["fav_goal", "fav_goal_position"]

def fmt_summary(vals):
    v = vals[~np.isnan(vals)]
    return f"{np.mean(v):.2f} \u00b1 {np.std(v):.2f} ({np.median(v):.2f})"

metric_names = list(prefixes.values())

# --- Build tables ---
desc_data = {}
stats_data = {}

for metric, metric_name in prefixes.items():
    for model in compare_models:
        display_name = model_names_dict[model]
        key_A = (model, vname_A)
        key_B = (model, vname_B)

        vals_A = plotters[key_A].metrics[metric].values.astype(float)
        vals_B = plotters[key_B].metrics[metric].values.astype(float)
        clean_A = vals_A[~np.isnan(vals_A)]
        clean_B = vals_B[~np.isnan(vals_B)]

        # Descriptives for both versions
        desc_data.setdefault((display_name, vname_A), {})[metric_name] = h.fmt_summary(vals_A)
        desc_data.setdefault((display_name, vname_B), {})[metric_name] = h.fmt_summary(vals_B)

        # MWU test (median comparison)
        U, p_mwu = stats.mannwhitneyu(clean_A, clean_B)
        stats_data.setdefault((display_name, "Md."), {})[metric_name] = f"{U:.1f}{h.stars(p_mwu)}"

        # Distribution test: Energy for multivariate learning, chi2 for categorical, KS for continuous
        if metric == "p_correct_learning":
            mat_A = np.column_stack([plotters[key_A].metrics[f"p_correct_b{b}"].values.astype(float) for b in range(2, 8)])
            mat_B = np.column_stack([plotters[key_B].metrics[f"p_correct_b{b}"].values.astype(float) for b in range(2, 8)])
            e_stat, p_dist = Energy().test(mat_A, mat_B)
            stats_data.setdefault((display_name, "Dist."), {})[metric_name] = f"{e_stat:.2f}{h.stars(p_dist)}"
        elif metric in cat_vars:
            all_cats = set(clean_A.astype(int)) | set(clean_B.astype(int))
            counts_A = np.array([np.sum(clean_A == c) for c in sorted(all_cats)])
            counts_B = np.array([np.sum(clean_B == c) for c in sorted(all_cats)])
            contingency = np.array([counts_A, counts_B])
            chi2, p_dist, dof, expected = stats.chi2_contingency(contingency)
            stats_data.setdefault((display_name, "Dist."), {})[metric_name] = f"{chi2:.2f}{h.stars(p_dist)}"
        else:
            ks_stat, p_dist = stats.ks_2samp(clean_A, clean_B)
            stats_data.setdefault((display_name, "Dist."), {})[metric_name] = f"{ks_stat:.3f}{h.stars(p_dist)}"

# --- Table 1: descriptives ---
desc_df = pd.DataFrame(desc_data, index=metric_names)
desc_df.columns = pd.MultiIndex.from_tuples(desc_df.columns)
desc_df.index.name = "Metric"
display(HTML(f"<h4>Descriptive statistics ({vname_A} vs {vname_B})</h4>"))
display(desc_df)

# --- Table 2: test statistics ---
stats_df = pd.DataFrame(stats_data, index=metric_names)
stats_df.columns = pd.MultiIndex.from_tuples(stats_df.columns)
stats_df.index.name = "Metric"
display(HTML(f"<h4>{vname_A} vs {vname_B} (* p&lt;.05, ** p&lt;.01, *** p&lt;.001)</h4>"))
display(stats_df)